In [ ]:
import os
import requests

# Configuración para la colección RIDIRP (Intelligent ROP System)
COLLECTION_ID = "6626162"
API_BASE_URL = "https://api.figshare.com/v2"
DOWNLOAD_DIR = "../../datasets/RIDIRP_Original"

In [2]:
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

In [3]:
def descargar_ridirp():
    print(f"Conectando a la colección {COLLECTION_ID} de Figshare...")
    
    # page_size=100 garantiza que capturemos todos los lotes de datos
    url_articles = f"{API_BASE_URL}/collections/{COLLECTION_ID}/articles?page_size=100"
    
    try:
        response_articles = requests.get(url_articles)
        response_articles.raise_for_status()
        articulos = response_articles.json()
        
        print(f"Se encontraron {len(articulos)} artículos en la colección.")

        for articulo in articulos:
            articulo_id = articulo['id']
            print(f"\nConsultando archivos para el artículo ID: {articulo_id}...")
            
            url_files = f"{API_BASE_URL}/articles/{articulo_id}/files"
            response_files = requests.get(url_files)
            response_files.raise_for_status()
            archivos = response_files.json()

            for archivo in archivos:
                nombre_archivo = archivo['name']
                url_descarga = archivo['download_url']
                ruta_local = os.path.join(DOWNLOAD_DIR, nombre_archivo)
                
                # Validación para continuar si se interrumpe la descarga
                if os.path.exists(ruta_local):
                    print(f"El archivo {nombre_archivo} ya existe localmente. Omitiendo...")
                    continue
                    
                tamano_mb = archivo['size'] / (1024 * 1024)
                print(f"Descargando {nombre_archivo} ({tamano_mb:.2f} MB)...")
                
                # Streaming en chunks para no saturar la RAM
                with requests.get(url_descarga, stream=True) as r:
                    r.raise_for_status()
                    with open(ruta_local, 'wb') as f:
                        for chunk in r.iter_content(chunk_size=8192):
                            f.write(chunk)
                            
                print(f"Guardado exitosamente: {ruta_local}")
                
    except requests.exceptions.RequestException as e:
        print(f"Error de conexión con la API: {e}")

In [4]:
descargar_ridirp()
print("\nProceso de descarga de RIDIRP finalizado.")

Conectando a la colección 6626162 de Figshare...
Se encontraron 4 artículos en la colección.

Consultando archivos para el artículo ID: 24565882...
Descargando infant_retinal_database_info.xlsx (0.03 MB)...
Guardado exitosamente: RIDIRP_Original\infant_retinal_database_info.xlsx

Consultando archivos para el artículo ID: 24565762...
Descargando images_stack.zip (2567.65 MB)...
Guardado exitosamente: RIDIRP_Original\images_stack.zip

Consultando archivos para el artículo ID: 24565819...
Descargando images_stack_without_captions.zip (2477.10 MB)...
Guardado exitosamente: RIDIRP_Original\images_stack_without_captions.zip

Consultando archivos para el artículo ID: 24565681...
Descargando images.zip (2567.75 MB)...
Guardado exitosamente: RIDIRP_Original\images.zip

Proceso de descarga de RIDIRP finalizado.
